In [2]:

# ============================================================
# SISTEM PAKAR, FUZZY INFERENCE SYSTEM (FIS), DAN XGBOOST
# DATASET A: Dataset_A_FIS_SistemPakar_200.csv
# DATASET B: Dataset_B_XGBoost_200.csv
# ============================================================

# 1. IMPORT LIBRARY
import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

print("=" * 65)
print("SISTEM PAKAR + FUZZY INFERENCE SYSTEM + XGBOOST")
print("=" * 65)


# ============================================================
# 2. MEMBACA DATASET
# ============================================================

df_a = pd.read_csv('Dataset_A_FIS_SistemPakar_200.csv')
df_b = pd.read_csv('Dataset_B_XGBoost_200.csv')

print("\nDATASET BERHASIL DIBACA")
print("-" * 65)

print("Dataset A:", df_a.shape)
print("Dataset B:", df_b.shape)

print("\nKolom Dataset A:")
print(df_a.columns.tolist())

print("\nKolom Dataset B:")
print(df_b.columns.tolist())


# ============================================================
# 3. IMPLEMENTASI FUZZY INFERENCE SYSTEM (FIS)
# ============================================================

print("\n" + "=" * 65)
print("PROSES FUZZY INFERENCE SYSTEM (FIS)")
print("=" * 65)

# Menghitung rata-rata dan standar deviasi populasi
mean_bb = df_a['BB_kg'].mean()
sd_bb = df_a['BB_kg'].std(ddof=0)

mean_tb = df_a['TB_cm'].mean()
sd_tb = df_a['TB_cm'].std(ddof=0)

if sd_bb == 0 or sd_tb == 0:
    raise ValueError("Standar deviasi BB atau TB bernilai 0.")

# Menghitung Z-Score
df_a['Z_BB'] = (df_a['BB_kg'] - mean_bb) / sd_bb
df_a['Z_TB'] = (df_a['TB_cm'] - mean_tb) / sd_tb


# Fungsi Fuzzy Inference System
def run_fis(row):

    # Membership function BB rendah
    mu_bb_rendah = (
        max(0, min(1, (-1.5 - row['Z_BB']) / 1.5))
        if row['Z_BB'] < 0 else 0
    )

    # Membership function BB normal
    mu_bb_normal = max(
        0, 1 - abs(row['Z_BB']) / 2.0
    )

    # Membership function TB rendah
    mu_tb_rendah = (
        max(0, min(1, (-1.5 - row['Z_TB']) / 1.5))
        if row['Z_TB'] < 0 else 0
    )

    # Membership function TB normal
    mu_tb_normal = max(
        0, 1 - abs(row['Z_TB']) / 2.0
    )

    # Operator Fuzzy
    w1 = max(mu_bb_rendah, mu_tb_rendah)
    w2 = min(mu_bb_normal, mu_tb_normal)

    # Defuzzifikasi
    if (w1 + w2) > 0:
        skor = (w1 * 85.0 + w2 * 15.0) / (w1 + w2)
    else:
        skor = 20.0

    return skor


# Menghitung skor FIS
df_a['Skor_FIS'] = df_a.apply(run_fis, axis=1)

# Menentukan hasil prediksi FIS
df_a['Pred_FIS'] = np.where(
    df_a['Skor_FIS'] >= 50.0, 1, 0
)

# Menampilkan hasil FIS
print("\nHASIL PERHITUNGAN FUZZY INFERENCE SYSTEM")
print("-" * 65)

print(df_a[
    ['BB_kg', 'TB_cm', 'Z_BB', 'Z_TB',
     'Skor_FIS', 'Pred_FIS']
].head(10).to_string(index=False))

print("\nJumlah Prediksi FIS:")
print(df_a['Pred_FIS'].value_counts().sort_index())


# ============================================================
# 4. IMPLEMENTASI EXPERT SYSTEM (SISTEM PAKAR)
# ============================================================

print("\n" + "=" * 65)
print("PROSES EXPERT SYSTEM (SISTEM PAKAR)")
print("=" * 65)

# Memastikan nama kolom sesuai dataset
lila_col = (
    'LILA_cm' if 'LILA_cm' in df_a.columns else 'LILA'
)

asi_col = (
    'ASI_eksklusif_6_bulan'
    if 'ASI_eksklusif_6_bulan' in df_a.columns
    else 'ASI'
)

# Memastikan kolom tersedia
required_es = ['Z_TB', 'Z_BB', lila_col, asi_col]

for col in required_es:
    if col not in df_a.columns:
        raise ValueError(
            f"Kolom {col} tidak ditemukan pada Dataset A."
        )


# Fungsi aturan Expert System
def run_expert_system(row):

    # Rule 1: Risiko stunting
    if row['Z_TB'] < -2.0:
        return 1

    # Rule 2: Risiko gizi kurang
    elif row['Z_BB'] < -2.0:
        return 1

    # Rule 3: Faktor risiko Non-ASI dan LILA rendah
    elif (
        row[asi_col] == 0
        and row[lila_col] < 11.5
    ):
        return 1

    # Rule default: Normal
    else:
        return 0


# Menghitung prediksi Expert System
df_a['Pred_ES'] = df_a.apply(
    run_expert_system, axis=1
)

# Menampilkan hasil Expert System
print("\nHASIL PREDIKSI EXPERT SYSTEM")
print("-" * 65)

print(df_a[
    ['BB_kg', 'TB_cm', lila_col, asi_col, 'Pred_ES']
].head(10).to_string(index=False))

print("\nJumlah Prediksi Expert System:")
print(df_a['Pred_ES'].value_counts().sort_index())


# ============================================================
# 5. PERBANDINGAN FIS DAN EXPERT SYSTEM
# ============================================================

print("\n" + "=" * 65)
print("PERBANDINGAN FIS DAN EXPERT SYSTEM")
print("=" * 65)

perbandingan = pd.crosstab(
    df_a['Pred_FIS'],
    df_a['Pred_ES'],
    rownames=['Prediksi FIS'],
    colnames=['Prediksi Expert System']
)

print(perbandingan)

print("\nPersentase Kesamaan Prediksi:")
kesamaan = (
    df_a['Pred_FIS'] == df_a['Pred_ES']
).mean() * 100

print(f"{kesamaan:.2f}%")


# ============================================================
# 6. PERSIAPAN DATASET B UNTUK XGBOOST
# ============================================================

print("\n" + "=" * 65)
print("PERSIAPAN DATASET XGBOOST")
print("=" * 65)

# Memastikan kolom target tersedia
if 'Label_risiko' not in df_b.columns:
    raise ValueError("Kolom Label_risiko tidak ditemukan.")

# Memisahkan fitur dan target
X = df_b.drop(
    columns=['ID', 'Nama', 'Label_risiko'],
    errors='ignore'
).copy()

# Mapping target menjadi numerik
label_mapping = {
    '0 (Normal)': 0,
    '1 (Risiko)': 1,
    'Normal': 0,
    'Risiko': 1,
    0: 0,
    1: 1
}

y = df_b['Label_risiko'].map(label_mapping)

# Memeriksa label yang tidak dikenali
if y.isna().any():
    nilai_tidak_dikenal = df_b.loc[
        y.isna(), 'Label_risiko'
    ].unique()

    raise ValueError(
        f"Label target tidak dikenali: {nilai_tidak_dikenal}"
    )

y = y.astype(int)

# Mapping fitur kategori
if 'Jenis_kelamin' in X.columns:
    X['Jenis_kelamin'] = X['Jenis_kelamin'].replace({
        'Laki-laki': 0,
        'Perempuan': 1,
        'L': 0,
        'P': 1
    })

if 'ASI_eksklusif_6_bulan' in X.columns:
    X['ASI_eksklusif_6_bulan'] = X[
        'ASI_eksklusif_6_bulan'
    ].replace({
        'Ya': 1,
        'Tidak': 0
    })

# Mengubah fitur kategori lain menjadi numerik
X = pd.get_dummies(
    X,
    drop_first=False,
    dtype=int
)

# Memastikan seluruh fitur numerik
X = X.apply(pd.to_numeric, errors='coerce')

# Menghapus kolom yang seluruh nilainya kosong
X = X.dropna(axis=1, how='all')

if X.shape[1] == 0:
    raise ValueError("Tidak ada fitur yang dapat digunakan.")

# Menghapus baris dengan nilai target kosong jika ada
valid_index = y.notna()

X = X.loc[valid_index]
y = y.loc[valid_index]

print("Jumlah data:", len(X))
print("Jumlah fitur:", X.shape[1])

print("\nDistribusi Label:")
print(y.value_counts().sort_index())

print("\nNama Fitur:")
print(X.columns.tolist())


# ============================================================
# 7. TRAIN TEST SPLIT (80% TRAINING, 20% TESTING)
# ============================================================

print("\n" + "=" * 65)
print("PEMBAGIAN DATA TRAINING DAN TESTING")
print("=" * 65)

# Memastikan setiap kelas cukup untuk stratifikasi
if y.value_counts().min() < 2:
    raise ValueError(
        "Jumlah anggota salah satu kelas terlalu sedikit."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data Training:", X_train.shape)
print("Data Testing :", X_test.shape)


# ============================================================
# 8. HYPERPARAMETER TUNING DAN CROSS VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("PROSES GRID SEARCH DAN 5-FOLD CROSS VALIDATION")
print("=" * 65)

# Lima hyperparameter
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Stratified 5-Fold
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Inisialisasi model
xgb = XGBClassifier(
    eval_metric='logloss',
    random_state=42,
    n_jobs=1
)

# GridSearchCV
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Training dan tuning
grid.fit(X_train, y_train)

# Model terbaik
best_model = grid.best_estimator_

print("\nPROSES TRAINING SELESAI")

print("\nBest Hyperparameters:")
print(grid.best_params_)

print("\nBest Cross Validation F1-Score:")
print(round(grid.best_score_, 4))


# ============================================================
# 9. PREDIKSI DATA TESTING XGBOOST
# ============================================================

print("\n" + "=" * 65)
print("PREDIKSI DATA TESTING XGBOOST")
print("=" * 65)

y_pred_xgb = best_model.predict(X_test)

y_prob_xgb = best_model.predict_proba(
    X_test
)[:, 1]

print("Prediksi berhasil dilakukan.")
print("Jumlah data testing:", len(y_test))


# ============================================================
# 10. EVALUASI MODEL XGBOOST
# ============================================================

print("\n" + "=" * 65)
print("HASIL EVALUASI MODEL XGBOOST")
print("=" * 65)

accuracy = accuracy_score(y_test, y_pred_xgb)

precision = precision_score(
    y_test, y_pred_xgb, zero_division=0
)

recall = recall_score(
    y_test, y_pred_xgb, zero_division=0
)

f1 = f1_score(
    y_test, y_pred_xgb, zero_division=0
)

auc = roc_auc_score(
    y_test, y_prob_xgb
) if len(np.unique(y_test)) == 2 else np.nan

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}" if not np.isnan(auc)
      else "ROC-AUC   : Tidak dapat dihitung")


# ============================================================
# 11. CONFUSION MATRIX
# ============================================================

print("\n" + "=" * 65)
print("CONFUSION MATRIX XGBOOST")
print("=" * 65)

cm = confusion_matrix(
    y_test,
    y_pred_xgb,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=['Aktual Normal (0)', 'Aktual Risiko (1)'],
    columns=['Prediksi Normal (0)', 'Prediksi Risiko (1)']
)

print(cm_df)


# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 65)
print("CLASSIFICATION REPORT")
print("=" * 65)

print(classification_report(
    y_test,
    y_pred_xgb,
    labels=[0, 1],
    target_names=['Normal', 'Risiko'],
    zero_division=0
))


# ============================================================
# 13. TABEL HASIL PREDIKSI XGBOOST
# ============================================================

print("\n" + "=" * 65)
print("TABEL HASIL PREDIKSI XGBOOST")
print("=" * 65)

hasil_prediksi = pd.DataFrame({
    'Label_Aktual': y_test.values,
    'Prediksi_XGBoost': y_pred_xgb,
    'Probabilitas_Risiko': y_prob_xgb
}, index=y_test.index)

print(hasil_prediksi.head(20).to_string())


# ============================================================
# 14. TABEL RINGKASAN EVALUASI
# ============================================================

print("\n" + "=" * 65)
print("RINGKASAN HASIL EVALUASI XGBOOST")
print("=" * 65)

evaluasi = pd.DataFrame({
    'Metrik': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1-Score',
        'ROC-AUC'
    ],
    'Nilai': [
        accuracy,
        precision,
        recall,
        f1,
        auc
    ]
})

print(evaluasi.to_string(index=False))


# ============================================================
# 15. MENYIMPAN HASIL KE FILE CSV
# ============================================================

# Hasil FIS dan Expert System
df_a.to_csv(
    'Hasil_FIS_Expert_System.csv',
    index=False
)

# Hasil prediksi XGBoost
hasil_prediksi.to_csv(
    'Hasil_Prediksi_XGBoost.csv',
    index=True
)

# Hasil evaluasi
evaluasi.to_csv(
    'Hasil_Evaluasi_XGBoost.csv',
    index=False
)

print("\n" + "=" * 65)
print("SELURUH PROSES BERHASIL DIJALANKAN")
print("=" * 65)

print("File yang berhasil disimpan:")
print("1. Hasil_FIS_Expert_System.csv")
print("2. Hasil_Prediksi_XGBoost.csv")
print("3. Hasil_Evaluasi_XGBoost.csv")

print("\nSELESAI!")


SISTEM PAKAR + FUZZY INFERENCE SYSTEM + XGBOOST

DATASET BERHASIL DIBACA
-----------------------------------------------------------------
Dataset A: (200, 9)
Dataset B: (200, 9)

Kolom Dataset A:
['ID', 'Nama', 'Umur_bulan', 'BB_kg', 'TB_cm', 'LILA_cm', 'Jenis_kelamin', 'ASI_eksklusif_6_bulan', 'Status_gizi']

Kolom Dataset B:
['ID', 'Nama', 'Umur_bulan', 'BB_kg', 'TB_cm', 'LILA_cm', 'Jenis_kelamin', 'ASI_eksklusif_6_bulan', 'Label_risiko']

PROSES FUZZY INFERENCE SYSTEM (FIS)

HASIL PERHITUNGAN FUZZY INFERENCE SYSTEM
-----------------------------------------------------------------
 BB_kg  TB_cm      Z_BB      Z_TB  Skor_FIS  Pred_FIS
  10.2   82.0  0.800093  0.735619      15.0         0
  11.8   88.0  1.252682  1.184099      15.0         0
   7.4   74.0  0.008062  0.137646      15.0         0
  15.1   96.0  2.186148  1.782073      20.0         0
   6.8   68.0 -0.161659 -0.310834      15.0         0
   9.5   80.0  0.602085  0.586126      15.0         0
  13.2   93.0  1.648698  1.5578